# Getting API Access

In [1]:
import hmac
import hashlib
import json
import os
import sys
import time
import requests
from dotenv import load_dotenv

load_dotenv()

def _require_env(name: str, *, as_int: bool = False):
    val = os.getenv(name)
    if not val:
        sys.exit(
            f"ERROR: environment variable {name} is not set. "
            f"Add it to your .env file before running this notebook."
        )
    if as_int:
        try:
            return int(val)
        except ValueError:
            sys.exit(f"ERROR: {name} must be an integer, got: {val!r}")
    return val


partner_id  = _require_env("SHOPEE_PARTNER_ID", as_int=True)
partner_key = _require_env("SHOPEE_PARTNER_KEY")

In [27]:
def shop_auth() -> str:
    """Generate the partner authorization URL and return it (also prints)."""
    timest = int(time.time())
    host = "https://partner.shopeemobile.com"
    path = "/api/v2/shop/auth_partner"
    redirect_url = "https://www.google.com/"
    tmp_base_string = f"{partner_id}{path}{timest}"
    base_string = tmp_base_string.encode()
    sign = hmac.new(partner_key.encode(), base_string, hashlib.sha256).hexdigest()
    url = (
        f"{host}{path}"
        f"?partner_id={partner_id}"
        f"&timestamp={timest}"
        f"&sign={sign}"
        f"&redirect={redirect_url}"
    )
    print(url)
    return url

In [28]:
auth_url = shop_auth()

https://partner.shopeemobile.com/api/v2/shop/auth_partner?partner_id=2013893&timestamp=1778757691&sign=16bb273f1fd0f24a16e2b466581f727a09a3f54ee568a37a752f9580e3438ebb&redirect=https://www.google.com/


In [29]:
# GetAccessToken
# After successful authorization, use the code and shop_id from the redirect URL
# to call this API. It returns shop_id, access_token, and refresh_token.
#
# https://partner.shopeemobile.com/api/v2/auth/token/get

In [2]:
shop_id = _require_env("SHOPEE_SHOP_ID", as_int=True)
code    = _require_env("SHOPEE_SHOP_CODE")


def get_token_shop_level(code, partner_id, tmp_partner_key, shop_id):
    """First-time token request after partner authorization."""
    timest = int(time.time())
    host = "https://partner.shopeemobile.com"
    path = "/api/v2/auth/token/get"
    body = {"code": code, "shop_id": shop_id, "partner_id": partner_id}
    tmp_base_string = f"{partner_id}{path}{timest}"
    sign = hmac.new(tmp_partner_key.encode(), tmp_base_string.encode(), hashlib.sha256).hexdigest()
    url = f"{host}{path}?partner_id={partner_id}&timestamp={timest}&sign={sign}"
    headers = {"Content-Type": "application/json"}

    try:
        resp = requests.post(url, json=body, headers=headers, timeout=30)
        ret = resp.json()
    except Exception as e:
        print(f"Request failed: {e}")
        return None, None

    if ret.get("error"):
        print(f"Error: {ret.get('error')} — {ret.get('message')}")
        return None, None

    access_token      = ret.get("access_token")
    new_refresh_token = ret.get("refresh_token")
    if not access_token or not new_refresh_token:
        print(f"Unexpected response shape: {json.dumps(ret, indent=2)}")
        return None, None

    return access_token, new_refresh_token

In [ ]:
access_token, refresh_token = get_token_shop_level(code, partner_id, partner_key, shop_id)
print(access_token)
print(refresh_token)

In [32]:
# RefreshAccessToken
# Before access_token expires, call this with refresh_token to get a new pair.
# The new refresh_token must be used the next time this API is called.
#
# https://partner.shopeemobile.com/api/v2/auth/access_token/get

In [3]:
access_token  = _require_env("SHOPEE_ACCESS_TOKEN")
refresh_token = _require_env("SHOPEE_REFRESH_TOKEN")


def get_access_token_shop_level(shop_id, partner_id, tmp_partner_key, refresh_token):
    """Refresh access_token using the current refresh_token."""
    timest = int(time.time())
    host = "https://partner.shopeemobile.com"
    path = "/api/v2/auth/access_token/get"
    body = {"shop_id": shop_id, "refresh_token": refresh_token, "partner_id": partner_id}
    tmp_base_string = f"{partner_id}{path}{timest}"
    sign = hmac.new(tmp_partner_key.encode(), tmp_base_string.encode(), hashlib.sha256).hexdigest()
    url = f"{host}{path}?partner_id={partner_id}&timestamp={timest}&sign={sign}"
    headers = {"Content-Type": "application/json"}

    try:
        resp = requests.post(url, json=body, headers=headers, timeout=30)
        ret = resp.json()
    except Exception as e:
        print(f"Request failed: {e}")
        return None, None
    if ret.get("error"):
        print(f"Error: {ret.get('error')} — {ret.get('message')}")
        return None, None

    print(json.dumps(ret, indent=4))

    access_token      = ret.get("access_token")
    new_refresh_token = ret.get("refresh_token")
    if not access_token or not new_refresh_token:
        print(f"Unexpected response shape: {json.dumps(ret, indent=2)}")
        return None, None

    return access_token, new_refresh_token

In [ ]:
access_token, refresh_token = get_access_token_shop_level(shop_id, partner_id, partner_key, refresh_token)
print(access_token)
print(refresh_token)